# MMGMT 721 Milestone 2 – Data wrangling pipeline

**Project:** USD/CAD exchange rate and Canadian energy exports to the U.S. · Study period: January 2018 – December 2025

This notebook is shared by the group. Each part starts with a heading **Part N – topic (name)**. Add your code and notes under your own part and run the notebook from the top (**Run All**) so that later parts can use the results of earlier parts.

| Part | Content | Owner |
|---|---|---|
| Part 2 | Data quality checks and cleaning | Rui |
| Part 3 | Data transformation and integration |    Daisy |
| Part 4 | Prepared dataset and data storage | Emily|




## Part 2 – Data quality checks and cleaning (Rui)


**Input**
- Statistics Canada Table 12-10-0182-01 (monthly exports, thousands of CAD)
- Bank of Canada daily exchange rate FXUSDCAD (CAD per 1 USD)

**Output**
- `data/clean/exports_clean.csv` and `data/clean/fx_daily_clean.csv` (optionally also saved to MotherDuck)
- `data/clean/quality_log.csv` and `docs/cleaning_log.md`, which are filled in automatically

**How to use:** check the settings in the next cell, then click **Run All**. Each check prints its result, and the cleaning log at the end is built from those results.

In [1]:
# ===================== SETTINGS (edit here) =====================
SOURCE = "csv"              # "csv" = read files from data/raw ; "motherduck" = read raw tables from MotherDuck
SAVE_TO_MOTHERDUCK = False  # True = also write the clean tables to MotherDuck (schema "clean")

# Raw files in data/raw/ (None = find automatically by name)
EXPORTS_CSV = None          # e.g. "12100182.csv"
FX_CSV = None               # e.g. "FXUSDCAD.csv"

# MotherDuck (only used when SOURCE = "motherduck" or SAVE_TO_MOTHERDUCK = True)
MD_DATABASE = "energy_fx"
RAW_EXPORTS_TABLE = "raw.exports"
RAW_FX_TABLE = "raw.fx_daily"

# Study period
START_MONTH = "2018-01"
END_MONTH = "2025-12"

# Which StatCan series to keep (matched case-insensitively)
TRADE_KEEP = "Export"              # StatCan label for total exports in this table
PRODUCT_CODE_PATTERN = r"\[(?:C12|14[1-6]|261)\]"   # Energy products [C12] and its 7 sub-groups [141]–[146], [261]
DEST_US = "United States"
DEST_ALL = "All countries"    # optional: only used if "All countries" was downloaded
ORIGIN_KEEP = "Canada"       # country of origin: Canada = goods produced in Canada

# Thresholds used to flag (not delete) unusual values
EXPORT_OUTLIER_PCT = 0.25   # month-over-month change in export value larger than ±25%
FX_JUMP_PCT = 0.02          # day-over-day change in the exchange rate larger than ±2%

In [2]:
from pathlib import Path
import io
import re

import numpy as np
import pandas as pd
import duckdb
from dotenv import load_dotenv

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR, CLEAN_DIR, DOCS_DIR = ROOT / "data" / "raw", ROOT / "data" / "clean", ROOT / "docs"
CLEAN_DIR.mkdir(parents=True, exist_ok=True)
DOCS_DIR.mkdir(parents=True, exist_ok=True)
load_dotenv(ROOT / ".env")
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

PERIOD_START = pd.Timestamp(START_MONTH + "-01")
PERIOD_END = pd.Timestamp(END_MONTH + "-01") + pd.offsets.MonthEnd(0)
N_MONTHS = len(pd.period_range(START_MONTH, END_MONTH, freq="M"))

LOG = []
def log(dataset, check, finding, decision, rationale):
    """Record one row of the cleaning log and print the finding."""
    LOG.append({"Dataset": dataset, "Check": check, "Finding": finding,
                "Decision": decision, "Rationale": rationale})
    print(f"[{dataset}] {check}: {finding}")

def find_file(patterns):
    for pattern in patterns:
        hits = sorted(RAW_DIR.glob(pattern))
        if hits:
            return hits[0]
    files = [f.name for f in RAW_DIR.iterdir() if f.name != ".gitkeep"]
    raise FileNotFoundError(f"No file matching {patterns} in {RAW_DIR}. Files found: {files}")

def find_col(df, *keywords):
    for c in df.columns:
        if all(k.lower() in c.lower() for k in keywords):
            return c
    return None

_con = None
def md_con():
    """Connect to MotherDuck (reads the motherduck_token from .env)."""
    global _con
    if _con is None:
        _con = duckdb.connect(f"md:{MD_DATABASE}")
    return _con

print("Project folder:", ROOT)
print(f"Study period: {START_MONTH} to {END_MONTH} ({N_MONTHS} months)")

Project folder: c:\Users\chenr\Projects\721\energy-fx
Study period: 2018-01 to 2025-12 (96 months)


### 2.1 Energy exports – Statistics Canada Table 12-10-0182-01

In [3]:
if SOURCE == "csv":
    exp_path = RAW_DIR / EXPORTS_CSV if EXPORTS_CSV else find_file(["*12100182*.csv", "*1210018201*.csv"])
    raw_exp = pd.read_csv(exp_path, dtype=str, encoding="utf-8-sig")
    print("Read file:", exp_path.name)
else:
    raw_exp = md_con().sql(f"SELECT * FROM {RAW_EXPORTS_TABLE}").df()
    print("Read MotherDuck table:", RAW_EXPORTS_TABLE)

raw_exp.columns = [c.strip() for c in raw_exp.columns]
print("Rows, columns:", raw_exp.shape)
raw_exp.head()

Read file: 1210018201_databaseLoadingData.csv
Rows, columns: (768, 18)


,REF_DATE,GEO,DGUID,North American Product Classification System (NAPCS),Trade,Principal trading partner: Exports by country of destination,Principal trading partner: Exports by country of origin,UOM,UOM_ID,SCALAR_FACTOR,SCALAR_ID,VECTOR,COORDINATE,VALUE,STATUS,SYMBOL,TERMINATED,DECIMALS
0,2018-01,Canada,2021A000011124,Energy products [C12],Export,"United States, country of destination","Canada, country of origin",Dollars,81,thousands,3,v1863565574,1.12.1.2.2,10020138,NaN,NaN,NaN,0
1,2018-02,Canada,2021A000011124,Energy products [C12],Export,"United States, country of destination","Canada, country of origin",Dollars,81,thousands,3,v1863565574,1.12.1.2.2,7772901,NaN,NaN,NaN,0
2,2018-03,Canada,2021A000011124,Energy products [C12],Export,"United States, country of destination","Canada, country of origin",Dollars,81,thousands,3,v1863565574,1.12.1.2.2,8436578,NaN,NaN,NaN,0
3,2018-04,Canada,2021A000011124,Energy products [C12],Export,"United States, country of destination","Canada, country of origin",Dollars,81,thousands,3,v1863565574,1.12.1.2.2,8343371,NaN,NaN,NaN,0
4,2018-05,Canada,2021A000011124,Energy products [C12],Export,"United States, country of destination","Canada, country of origin",Dollars,81,thousands,3,v1863565574,1.12.1.2.2,9491938,NaN,NaN,NaN,0


In [4]:
# Identify the dimension columns (names can differ slightly between downloads)
COL_TRADE = "Trade" if "Trade" in raw_exp.columns else find_col(raw_exp, "trade")
COL_PRODUCT = find_col(raw_exp, "napcs") or find_col(raw_exp, "product")
COL_DEST = find_col(raw_exp, "destination")
COL_ORIGIN = find_col(raw_exp, "origin")
for name, col in [("Trade", COL_TRADE), ("Product", COL_PRODUCT), ("Destination", COL_DEST), ("Origin", COL_ORIGIN)]:
    print(f"{name:12s} -> {col}")
assert COL_TRADE and COL_PRODUCT and COL_DEST, "A key column was not found: check the column names printed above."

# Show the categories available, so the filters in SETTINGS can be checked
for col in [COL_TRADE, COL_DEST, COL_ORIGIN]:
    if col:
        print(f"\n{col}:", sorted(raw_exp[col].dropna().astype(str).unique())[:40])
energy_products = sorted(v for v in raw_exp[COL_PRODUCT].dropna().astype(str).unique() if re.search(PRODUCT_CODE_PATTERN, v))
print(f"\nEnergy product categories found ({len(energy_products)}):", energy_products)

Trade        -> Trade
Product      -> North American Product Classification System (NAPCS)
Destination  -> Principal trading partner: Exports by country of destination
Origin       -> Principal trading partner: Exports by country of origin

Trade: ['Export']

Principal trading partner: Exports by country of destination: ['United States, country of destination']

Principal trading partner: Exports by country of origin: ['Canada, country of origin']

Energy product categories found (8): ['Coal [144]', 'Crude oil and bitumen [141]', 'Electricity [146]', 'Energy products [C12]', 'Natural gas [142]', 'Natural gas liquids (including condensate) and related products [143]', 'Nuclear fuel and other energy products [145]', 'Refined petroleum energy products (including liquid biofuels) [261]']


#### 2.1.1 Data types

In [5]:
exp = raw_exp.copy()
exp["month"] = pd.to_datetime(exp["REF_DATE"].astype(str).str[:7], format="%Y-%m", errors="coerce")
exp["value"] = pd.to_numeric(exp["VALUE"], errors="coerce")

bad_dates = int(exp["month"].isna().sum())
print(raw_exp[["REF_DATE", "VALUE"]].dtypes)
log("Exports", "Data types",
    f"REF_DATE and VALUE were read as {raw_exp['REF_DATE'].dtype} / {raw_exp['VALUE'].dtype}; {bad_dates} dates could not be parsed",
    "Converted REF_DATE to a monthly date (first day of month) and VALUE to a number",
    "Dates and numbers need the correct type for sorting, joining with the exchange-rate data and calculating changes")

REF_DATE    str
VALUE       str
dtype: object
[Exports] Data types: REF_DATE and VALUE were read as str / str; 0 dates could not be parsed


#### 2.1.2 Keep only the series needed for the research question

In [6]:
n_raw = len(exp)
in_period = exp["month"].between(PERIOD_START, PERIOD_END)
is_trade = exp[COL_TRADE].astype(str).str.strip().str.lower() == TRADE_KEEP.lower()
is_product = exp[COL_PRODUCT].astype(str).str.contains(PRODUCT_CODE_PATTERN, regex=True, na=False)
# StatCan labels look like "United States, country of destination": compare only the part before the comma
dest_short = exp[COL_DEST].astype(str).str.split(",").str[0].str.strip()
is_dest = dest_short.str.lower().isin([DEST_US.lower(), DEST_ALL.lower()])

for label, m in [("trade type", is_trade), ("energy products", is_product), ("destination", is_dest), ("study period", in_period)]:
    if not m.any():
        raise ValueError(f"No rows match the {label} filter. Check the SETTINGS cell against the categories printed above.")

mask = in_period & is_trade & is_product & is_dest
if COL_ORIGIN:
    origin_short = exp[COL_ORIGIN].astype(str).str.split(",").str[0].str.strip()
    is_origin = origin_short.str.lower() == ORIGIN_KEEP.lower()
    if not is_origin.any():
        raise ValueError(f"No rows match ORIGIN_KEEP = {ORIGIN_KEEP!r}. Options: {sorted(origin_short.unique())}")
    mask &= is_origin
    print("Country of origin kept:", ORIGIN_KEEP)

dropped_trade = sorted(set(exp.loc[in_period & is_product & is_dest & ~is_trade, COL_TRADE].astype(str)))
exp = exp[mask].copy()
exp["product_code"] = exp[COL_PRODUCT].astype(str).str.extract(r"\[([^\]]+)\]\s*$", expand=False)
exp["product_name"] = exp[COL_PRODUCT].astype(str).str.replace(r"\s*\[.*\]\s*$", "", regex=True)
exp["destination"] = exp[COL_DEST].astype(str).str.split(",").str[0].str.strip()

log("Exports", "Series selection",
    f"{n_raw:,} raw rows reduced to {len(exp):,} rows; trade types dropped: {dropped_trade or 'none'}",
    f"Kept exports (Trade = '{TRADE_KEEP}') of Energy products [C12] and its {exp['product_code'].nunique() - 1} sub-groups, "
    f"country of origin = {ORIGIN_KEEP}, destination = {', '.join(sorted(exp['destination'].unique()))}, {START_MONTH} to {END_MONTH}",
    "Canadian-origin goods measure energy produced in Canada (foreign goods re-exported through Canada are excluded); "
    "one trade measure is kept so nothing is double counted; other products, partners and months are outside the research question")
exp[["month", "product_code", "product_name", "destination", "value"]].head()

Country of origin kept: Canada
[Exports] Series selection: 768 raw rows reduced to 768 rows; trade types dropped: none


,month,product_code,product_name,destination,value
0,2018-01-01,C12,Energy products,United States,10020138
1,2018-02-01,C12,Energy products,United States,7772901
2,2018-03-01,C12,Energy products,United States,8436578
3,2018-04-01,C12,Energy products,United States,8343371
4,2018-05-01,C12,Energy products,United States,9491938


#### 2.1.3 Units

In [7]:
scalar = sorted(exp["SCALAR_FACTOR"].dropna().astype(str).unique()) if "SCALAR_FACTOR" in exp else ["(column not found)"]
uom = sorted(exp["UOM"].dropna().astype(str).unique()) if "UOM" in exp else ["(column not found)"]
log("Exports", "Units",
    f"Unit of measure: {uom}; scalar factor: {scalar}",
    "Kept values in thousands of Canadian dollars and named the column export_value_cad_k",
    "Keeps the published precision; the unit is recorded in the variable name and the data dictionary")

[Exports] Units: Unit of measure: ['Dollars']; scalar factor: ['thousands']


#### 2.1.4 Missing values and quality flags

In [8]:
SYMBOLS = {"..": "not available", "x": "suppressed (confidentiality)", "E": "use with caution",
           "F": "too unreliable to be published", "p": "preliminary", "r": "revised"}

n_missing = int(exp["value"].isna().sum())
flag_cols = [c for c in ["STATUS", "SYMBOL"] if c in exp.columns]
flags = {}
for c in flag_cols:
    vc = exp[c].dropna().astype(str).str.strip()
    vc = vc[vc != ""].value_counts()
    flags.update({f"{c}={k} ({SYMBOLS.get(k, 'see StatCan legend')})": int(v) for k, v in vc.items()})
print("Missing values:", n_missing)
print("Quality flags:", flags or "none")
if n_missing:
    display(exp.loc[exp["value"].isna(), ["month", "product_code", "destination", "VALUE"] + flag_cols])

log("Exports", "Missing values",
    f"{n_missing} missing export values",
    "None to fix" if n_missing == 0 else "Kept as missing (not filled with zero) and listed above",
    "A missing value means the figure was not published; replacing it with zero would understate exports")
log("Exports", "Quality flags (STATUS / SYMBOL)",
    f"{flags if flags else 'No flags'}",
    "Kept all values and carried the flag into the clean table" if flags else "No action needed",
    "Flagged values are still the official figures; the flag lets later steps treat them with caution")

Missing values: 0
Quality flags: none
[Exports] Missing values: 0 missing export values
[Exports] Quality flags (STATUS / SYMBOL): No flags


#### 2.1.5 Duplicates and completeness

In [9]:
key = ["month", "product_code", "destination"]
n_dup = int(exp.duplicated(subset=key).sum())
exp = exp.drop_duplicates(subset=key, keep="first")
log("Exports", "Duplicate rows",
    f"{n_dup} duplicate month-product-destination rows",
    "None to fix" if n_dup == 0 else "Removed duplicates, keeping the first row",
    "Each series must have exactly one value per month")

counts = exp.groupby(["product_code", "destination"])["month"].nunique()
incomplete = counts[counts != N_MONTHS]
print(counts.to_string())
log("Exports", "Completeness",
    f"{len(counts)} series; {len(counts) - len(incomplete)} have all {N_MONTHS} months" +
    (f"; incomplete: {incomplete.to_dict()}" if len(incomplete) else ""),
    "No gaps to fix" if len(incomplete) == 0 else "Listed the incomplete series for review",
    "Month-over-month changes need an unbroken monthly series")

[Exports] Duplicate rows: 0 duplicate month-product-destination rows
product_code  destination  
141           United States    96
142           United States    96
143           United States    96
144           United States    96
145           United States    96
146           United States    96
261           United States    96
C12           United States    96
[Exports] Completeness: 8 series; 8 have all 96 months


#### 2.1.6 Consistency checks
- Exports to the U.S. should never be larger than exports to all countries.
- The six energy sub-groups (C121–C126) should add up to the energy total (C12).

In [10]:
wide = exp.pivot_table(index=["month", "product_code"], columns="destination", values="value", aggfunc="first")
if DEST_US in wide.columns and DEST_ALL not in wide.columns:
    print(f"'{DEST_ALL}' not downloaded: U.S. vs all-countries check skipped")
if DEST_US in wide.columns and DEST_ALL in wide.columns:
    n_bad = int((wide[DEST_US] > wide[DEST_ALL] + 0.5).sum())
    log("Exports", "U.S. vs all countries",
        f"{n_bad} cases where U.S. exports exceed exports to all countries",
        "None to fix" if n_bad == 0 else "Listed the cases for review",
        "The U.S. is one of the destinations included in 'all countries'")

subs = exp[exp["product_code"] != "C12"]                # the sub-groups of Energy products
total = exp[exp["product_code"] == "C12"]
if len(subs) and len(total):
    s = subs.groupby(["month", "destination"])["value"].sum(min_count=1)
    t = total.set_index(["month", "destination"])["value"]
    gap = ((s - t).abs() / t).dropna()
    print(f"Largest gap between sum of sub-groups and C12 total: {gap.max():.4%}")
    log("Exports", "Sub-groups add up to total",
        f"{subs['product_code'].nunique()} sub-groups; largest gap vs C12 total: {gap.max():.3%}",
        "Accepted (gap caused by rounding only)" if gap.max() < 0.01 else "Gaps above 1% listed for review",
        "Confirms the product categories are defined consistently across the whole period")
else:
    print("Sub-groups not downloaded: check skipped")

'All countries' not downloaded: U.S. vs all-countries check skipped
Largest gap between sum of sub-groups and C12 total: 0.0000%
[Exports] Sub-groups add up to total: 7 sub-groups; largest gap vs C12 total: 0.000%


#### 2.1.7 Unusual month-over-month changes (flagged, not removed)

In [11]:
us_total = total[total["destination"] == DEST_US].sort_values("month").set_index("month")["value"]
pct = us_total.pct_change()
big = pct[pct.abs() > EXPORT_OUTLIER_PCT]
print(big.map("{:+.1%}".format).to_string() if len(big) else "No large changes")
log("Exports", "Unusual changes (energy exports to U.S.)",
    f"{len(big)} months with a change larger than ±{EXPORT_OUTLIER_PCT:.0%}: " +
    (", ".join(f"{d:%Y-%m} ({v:+.0%})" for d, v in big.items()) if len(big) else "none"),
    "Kept all values and flagged these months",
    "Checked against known events (for example the 2020 oil-price collapse); genuine market movements are kept so the analysis reflects real volatility")

if "TERMINATED" in exp.columns:
    n_term = int(exp["TERMINATED"].notna().sum() - (exp["TERMINATED"].astype(str).str.strip() == "").sum())
    log("Exports", "Terminated series", f"{n_term} rows from terminated series",
        "None to fix" if n_term == 0 else "Listed for review", "A terminated series would stop being updated")

month
2019-03-01    +33.9%
2020-04-01    -37.0%
2020-05-01    -26.7%
2020-07-01    +49.3%
2022-03-01    +25.0%
2025-12-01    +26.4%
[Exports] Unusual changes (energy exports to U.S.): 6 months with a change larger than ±25%: 2019-03 (+34%), 2020-04 (-37%), 2020-05 (-27%), 2020-07 (+49%), 2022-03 (+25%), 2025-12 (+26%)
[Exports] Terminated series: 0 rows from terminated series


In [12]:
exports_clean = (exp.assign(status=exp[flag_cols].fillna("").astype(str).agg(" ".join, axis=1).str.strip() if flag_cols else "")
                   .rename(columns={"value": "export_value_cad_k"})
                   [["month", "product_code", "product_name", "destination", "export_value_cad_k", "status"]]
                   .sort_values(["product_code", "destination", "month"])
                   .reset_index(drop=True))
print(exports_clean.shape)
exports_clean.head(10)

(768, 6)


,month,product_code,product_name,destination,export_value_cad_k,status
0,2018-01-01,141,Crude oil and bitumen,United States,7096708,
1,2018-02-01,141,Crude oil and bitumen,United States,5755592,
2,2018-03-01,141,Crude oil and bitumen,United States,6342070,
3,2018-04-01,141,Crude oil and bitumen,United States,6519028,
4,2018-05-01,141,Crude oil and bitumen,United States,7782490,
5,2018-06-01,141,Crude oil and bitumen,United States,7965624,
6,2018-07-01,141,Crude oil and bitumen,United States,8168446,
7,2018-08-01,141,Crude oil and bitumen,United States,8411045,
8,2018-09-01,141,Crude oil and bitumen,United States,6992975,
9,2018-10-01,141,Crude oil and bitumen,United States,7691990,


### 2.2 Exchange rate – Bank of Canada daily FXUSDCAD

In [13]:
def read_valet_csv(path):
    """Bank of Canada Valet CSVs start with notes; the data begins after the line "OBSERVATIONS"."""
    text = path.read_text(encoding="utf-8-sig")
    lines = text.splitlines()
    start = next((i for i, l in enumerate(lines) if l.strip().strip('"') == "OBSERVATIONS"), None)
    body = "\n".join(lines[start + 1:]) if start is not None else text
    return pd.read_csv(io.StringIO(body), dtype=str), start is not None

if SOURCE == "csv":
    fx_path = RAW_DIR / FX_CSV if FX_CSV else find_file(["*FXUSDCAD*.csv", "*fx*.csv", "*FX*.csv"])
    raw_fx, had_notes = read_valet_csv(fx_path)
    print("Read file:", fx_path.name)
else:
    raw_fx, had_notes = md_con().sql(f"SELECT * FROM {RAW_FX_TABLE}").df(), False
    print("Read MotherDuck table:", RAW_FX_TABLE)

raw_fx.columns = [c.strip() for c in raw_fx.columns]
fx_date_col = find_col(raw_fx, "date")
fx_val_col = "FXUSDCAD" if "FXUSDCAD" in raw_fx.columns else [c for c in raw_fx.columns if c != fx_date_col][0]
print("Date column:", fx_date_col, "| Rate column:", fx_val_col, "| Rows:", len(raw_fx))
if had_notes:
    log("Exchange rate", "File format", "The CSV starts with notes (terms and series description) before the data",
        "Read only the rows after the 'OBSERVATIONS' line", "The notes are not data and would break the table structure")
raw_fx.head()

Read file: FXUSDCAD.csv
Date column: date | Rate column: FXUSDCAD | Rows: 1995
[Exchange rate] File format: The CSV starts with notes (terms and series description) before the data


,date,FXUSDCAD
0,2018-01-02,1.2517
1,2018-01-03,1.2533
2,2018-01-04,1.2515
3,2018-01-05,1.2403
4,2018-01-08,1.2422


In [14]:
fx = pd.DataFrame({"date": pd.to_datetime(raw_fx[fx_date_col], errors="coerce"),
                   "usd_cad": pd.to_numeric(raw_fx[fx_val_col], errors="coerce")})
log("Exchange rate", "Data types",
    f"Date and rate were read as {raw_fx[fx_date_col].dtype} / {raw_fx[fx_val_col].dtype}; {int(fx['date'].isna().sum())} dates could not be parsed",
    "Converted the date to a date type and the rate to a number",
    "Needed to calculate monthly averages and join with the export data")

n_before = len(fx)
fx = fx[fx["date"].between(PERIOD_START, PERIOD_END)].copy()
log("Exchange rate", "Study period",
    f"{n_before} rows read; {len(fx)} in {START_MONTH} to {END_MONTH} (first {fx['date'].min():%Y-%m-%d}, last {fx['date'].max():%Y-%m-%d})",
    "Kept only dates inside the study period", "Matches the period of the export data")

n_missing_fx = int(fx["usd_cad"].isna().sum())
fx = fx.dropna(subset=["usd_cad"])
log("Exchange rate", "Missing values",
    f"{n_missing_fx} listed dates without a rate",
    "None to fix" if n_missing_fx == 0 else "Removed these rows (no value was published)",
    "Monthly averages should use only published rates; a blank must not count as zero")

n_dup_fx = int(fx.duplicated(subset="date").sum())
fx = fx.drop_duplicates(subset="date", keep="first").sort_values("date").reset_index(drop=True)
log("Exchange rate", "Duplicate dates", f"{n_dup_fx} duplicate dates",
    "None to fix" if n_dup_fx == 0 else "Removed duplicates, keeping the first", "One rate per business day")

[Exchange rate] Data types: Date and rate were read as str / str; 0 dates could not be parsed
[Exchange rate] Study period: 1995 rows read; 1995 in 2018-01 to 2025-12 (first 2018-01-02, last 2025-12-31)
[Exchange rate] Missing values: 0 listed dates without a rate
[Exchange rate] Duplicate dates: 0 duplicate dates


In [15]:
n_weekend = int((fx["date"].dt.dayofweek >= 5).sum())
days_per_month = fx.groupby(fx["date"].dt.to_period("M")).size()
missing_months = sorted(set(pd.period_range(START_MONTH, END_MONTH, freq="M")) - set(days_per_month.index))
log("Exchange rate", "Weekends, holidays and coverage",
    f"{n_weekend} weekend dates; {len(days_per_month)} of {N_MONTHS} months covered; "
    f"{days_per_month.min()}–{days_per_month.max()} business days per month" +
    (f"; months missing: {[str(m) for m in missing_months]}" if missing_months else ""),
    "Weekends and holidays not treated as missing; monthly averages will use business days only",
    "The Bank of Canada does not publish rates on weekends and statutory holidays")

print(f"Rate range: {fx['usd_cad'].min():.4f} to {fx['usd_cad'].max():.4f} CAD per USD")
n_nonpos = int((fx["usd_cad"] <= 0).sum())
jumps = fx.set_index("date")["usd_cad"].pct_change()
jumps = jumps[jumps.abs() > FX_JUMP_PCT]
log("Exchange rate", "Valid range and sudden jumps",
    f"Range {fx['usd_cad'].min():.4f}–{fx['usd_cad'].max():.4f}; {n_nonpos} zero/negative values; "
    f"{len(jumps)} daily moves larger than ±{FX_JUMP_PCT:.0%}" +
    (": " + ", ".join(f"{d:%Y-%m-%d} ({v:+.1%})" for d, v in jumps.items()) if len(jumps) else ""),
    "Kept all values; large moves flagged",
    "All values are plausible exchange rates; large moves reflect real market events")
log("Exchange rate", "Unit and direction", "Rate is Canadian dollars per 1 U.S. dollar",
    "Named the column usd_cad and recorded the unit",
    "A higher value means a weaker Canadian dollar; this must be clear when interpreting results")

fx_daily_clean = fx
fx_daily_clean.head()

[Exchange rate] Weekends, holidays and coverage: 0 weekend dates; 96 of 96 months covered; 19–23 business days per month
Rate range: 1.2040 to 1.4603 CAD per USD
[Exchange rate] Valid range and sudden jumps: Range 1.2040–1.4603; 0 zero/negative values; 1 daily moves larger than ±2%: 2020-03-18 (+2.3%)
[Exchange rate] Unit and direction: Rate is Canadian dollars per 1 U.S. dollar


C:\Users\chenr\AppData\Local\Temp\ipykernel_4564\2566171535.py:1: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  n_weekend = int((fx["date"].dt.dayofweek >= 5).sum())


,date,usd_cad
0,2018-01-02,1.2517
1,2018-01-03,1.2533
2,2018-01-04,1.2515
3,2018-01-05,1.2403
4,2018-01-08,1.2422


### 2.3 Save the clean data and the cleaning log

In [16]:
quality_log = pd.DataFrame(LOG)
exports_clean.to_csv(CLEAN_DIR / "exports_clean.csv", index=False)
fx_daily_clean.to_csv(CLEAN_DIR / "fx_daily_clean.csv", index=False)
quality_log.to_csv(CLEAN_DIR / "quality_log.csv", index=False)

def to_md_table(df):
    esc = lambda x: str(x).replace("|", "\\|").replace("\n", " ")
    head = "| " + " | ".join(df.columns) + " |\n|" + "---|" * len(df.columns) + "\n"
    return head + "".join("| " + " | ".join(esc(v) for v in row) + " |\n" for row in df.itertuples(index=False))

(DOCS_DIR / "cleaning_log.md").write_text(
    "# Cleaning log\n\nGenerated by Part 2 of `notebooks/milestone2_pipeline.ipynb`. "
    f"Study period: {START_MONTH} to {END_MONTH}.\n\n" + to_md_table(quality_log), encoding="utf-8")

if SAVE_TO_MOTHERDUCK:
    con = md_con()
    con.execute("CREATE SCHEMA IF NOT EXISTS clean")
    for name, df in [("exports", exports_clean), ("fx_daily", fx_daily_clean), ("quality_log", quality_log)]:
        con.register("tmp_df", df)
        con.execute(f"CREATE OR REPLACE TABLE clean.{name} AS SELECT * FROM tmp_df")
        con.unregister("tmp_df")
    print("Saved to MotherDuck:", MD_DATABASE, "(clean.exports, clean.fx_daily, clean.quality_log)")

print("Saved: data/clean/exports_clean.csv, data/clean/fx_daily_clean.csv, data/clean/quality_log.csv, docs/cleaning_log.md")
quality_log

Saved: data/clean/exports_clean.csv, data/clean/fx_daily_clean.csv, data/clean/quality_log.csv, docs/cleaning_log.md


,Dataset,Check,Finding,Decision,Rationale
0,Exports,Data types,REF_DATE and VALUE were read as str / str; 0 d...,Converted REF_DATE to a monthly date (first da...,Dates and numbers need the correct type for so...
1,Exports,Series selection,768 raw rows reduced to 768 rows; trade types ...,Kept exports (Trade = 'Export') of Energy prod...,Canadian-origin goods measure energy produced ...
2,Exports,Units,Unit of measure: ['Dollars']; scalar factor: [...,Kept values in thousands of Canadian dollars a...,Keeps the published precision; the unit is rec...
3,Exports,Missing values,0 missing export values,None to fix,A missing value means the figure was not publi...
4,Exports,Quality flags (STATUS / SYMBOL),No flags,No action needed,Flagged values are still the official figures;...
5,Exports,Duplicate rows,0 duplicate month-product-destination rows,None to fix,Each series must have exactly one value per month
6,Exports,Completeness,8 series; 8 have all 96 months,No gaps to fix,Month-over-month changes need an unbroken mont...
7,Exports,Sub-groups add up to total,7 sub-groups; largest gap vs C12 total: 0.000%,Accepted (gap caused by rounding only),Confirms the product categories are defined co...
8,Exports,Unusual changes (energy exports to U.S.),6 months with a change larger than ±25%: 2019-...,Kept all values and flagged these months,Checked against known events (for example the ...
9,Exports,Terminated series,0 rows from terminated series,None to fix,A terminated series would stop being updated


#### Hand-off to Part 3

Part 2 produces two clean tables that later parts can use directly (they are also saved in `data/clean/`):

| Variable | Rows | Columns | Notes |
|---|---|---|---|
| `exports_clean` | 768 (8 series × 96 months) | `month`, `product_code`, `product_name`, `destination`, `export_value_cad_k`, `status` | Long format; `product_code` = `C12` is the energy total, the other 7 codes are sub-groups; values in thousands of CAD |
| `fx_daily_clean` | 1,995 business days | `date`, `usd_cad` | Canadian dollars per 1 U.S. dollar; no weekends or holidays |

---
## Part 3 – Data transformation and integration (Daisy)

**Inputs:** `exports_clean` and `fx_daily_clean` produced by Part 2. Run Part 2 first.

This part retains Part 2's source selection: **Trade = Export; country of origin = Canada;
destination = United States; January 2018–December 2025**. The source label is not
renamed to Domestic exports. Export amounts remain in **thousands of current CAD**.

The workflow is daily FX → monthly average → join by month → calculate percentage changes
→ validate → save intermediate tables for Part 4. The original Part 2 DataFrames are not modified.
`C12` is the energy total; `141`–`146` and `261` are seven sub-groups. Never add C12 to its sub-groups.


### 3.1 Prepare the inputs

Part 2 has already cleaned the data and set the date and numeric types. Run Part 2 first,
then work on copies here. Sort each product–destination series by month and confirm its
calendar is complete before calculating changes. The dates already use the first day
of each month. Blank `status` means no published quality flag.


In [17]:
from pathlib import Path
import pandas as pd

# Use the clean tables and settings already created by Part 2.
p3_exports = exports_clean.copy()
p3_fx = fx_daily_clean.copy()
expected_months = pd.date_range(START_MONTH + '-01', END_MONTH + '-01', freq='MS')
p3_keys = ['month', 'product_code', 'destination']
p3_exports = p3_exports.sort_values(['product_code', 'destination', 'month']).reset_index(drop=True)

# A missing or duplicate month would make shift(1) or shift(12) compare the wrong rows.
for series_key, series in p3_exports.groupby(['product_code', 'destination']):
    assert pd.DatetimeIndex(series['month']).equals(expected_months), f'Check monthly coverage: {series_key}'

print(f"Inputs: {len(p3_exports):,} export rows; {len(p3_fx):,} daily FX observations.")


Inputs: 768 export rows; 1,995 daily FX observations.


### 3.2 Aggregate daily exchange rates to monthly averages

For each month, calculate the arithmetic mean of the published daily rates and retain
the observation count. Do not create weekend/holiday rates or replace them with zero.
`usd_cad_monthly_avg` is **CAD per USD**: an increase means Canadian-dollar depreciation.
Keep full precision for calculations; round only the displayed preview.


In [18]:
p3_fx['month'] = p3_fx['date'].dt.to_period('M').dt.to_timestamp()
fx_monthly = p3_fx.groupby('month', as_index=False).agg(
    usd_cad_monthly_avg=('usd_cad', 'mean'),
    fx_observation_count=('usd_cad', 'count')
)
fx_monthly = fx_monthly.sort_values('month').reset_index(drop=True)
assert pd.DatetimeIndex(fx_monthly['month']).equals(expected_months), 'Check missing or unexpected FX months.'
fx_monthly['fx_mom_pct'] = fx_monthly['usd_cad_monthly_avg'].pct_change(fill_method=None) * 100
display(fx_monthly.head().round({'usd_cad_monthly_avg': 6, 'fx_mom_pct': 4}))


,month,usd_cad_monthly_avg,fx_observation_count,fx_mom_pct
0,2018-01-01,1.242741,22,NaN
1,2018-02-01,1.258621,19,1.2778
2,2018-03-01,1.293171,21,2.7451
3,2018-04-01,1.273257,21,-1.5400
4,2018-05-01,1.287327,22,1.1051


### 3.3 Integrate exports and exchange rates

Merge on `month` with **many-to-one validation**: multiple products share one monthly
exchange rate. A left join retains every export row and its quality flag. Step 3.5 checks
for unmatched FX values and changes in the original export data. `exports_monthly` retains
all series; `analysis_monthly` will contain only the C12 energy total exported to the U.S.


In [19]:
exports_monthly = p3_exports.merge(
    fx_monthly, on='month', how='left', validate='many_to_one'
)
exports_monthly = exports_monthly.sort_values(['product_code', 'destination', 'month']).reset_index(drop=True)

# Keep the source selection visible for the data dictionary.
exports_monthly['trade_type'] = TRADE_KEEP
exports_monthly['country_of_origin'] = ORIGIN_KEEP
print(f"Monthly export table: {len(exports_monthly):,} rows.")


Monthly export table: 768 rows.


### 3.4 Calculate month-over-month and year-over-year changes

- **MoM (%)** = `(current / previous month - 1) × 100`.
- **YoY (%)** = `(current / same month last year - 1) × 100`.

Export changes are calculated **within each product and destination**, after sorting by month.
The `_pct` columns store percentage values: **2.5 means 2.5%**, not 250%.
No forward filling is used. With the present 2018–2025 inputs, the first MoM and the first
12 YoY values in each series are undefined and remain missing. A zero denominator also
produces a missing change, not infinity. `mom_pair_available` identifies usable pairs for Q3.


In [20]:
grouped_exports = exports_monthly.groupby(['product_code', 'destination'])['export_value_cad_k']

# shift(1): the previous month's value for the same product and destination.
previous_month = grouped_exports.shift(1)
previous_month = previous_month.where(previous_month != 0)
exports_monthly['exports_mom_pct'] = (exports_monthly['export_value_cad_k'] / previous_month - 1) * 100

# shift(12): the value in the same month one year earlier. Zero denominators stay missing.
previous_year = grouped_exports.shift(12)
previous_year = previous_year.where(previous_year != 0)
exports_monthly['exports_yoy_pct'] = (exports_monthly['export_value_cad_k'] / previous_year - 1) * 100

exports_monthly['mom_pair_available'] = exports_monthly[['fx_mom_pct', 'exports_mom_pct']].notna().all(axis=1)

# C12 is already the energy total: select it rather than adding it to its sub-groups.
is_us_total = (exports_monthly['product_code'] == 'C12') & (exports_monthly['destination'] == DEST_US)
analysis_monthly = exports_monthly.loc[is_us_total].sort_values('month').reset_index(drop=True).copy()

preview_columns = ['month', 'export_value_cad_k', 'usd_cad_monthly_avg',
                   'fx_mom_pct', 'exports_mom_pct', 'exports_yoy_pct']
display(analysis_monthly[preview_columns].head(3).round(
    {'usd_cad_monthly_avg': 6, 'fx_mom_pct': 4, 'exports_mom_pct': 4, 'exports_yoy_pct': 4}))


,month,export_value_cad_k,usd_cad_monthly_avg,fx_mom_pct,exports_mom_pct,exports_yoy_pct
0,2018-01-01,10020138,1.242741,NaN,NaN,NaN
1,2018-02-01,7772901,1.258621,1.2778,-22.4272,NaN
2,2018-03-01,8436578,1.293171,2.7451,8.5383,NaN


### 3.5 Validate the hand-off

The earlier steps check the monthly calendars and the merge relationship. Here, five
final checks confirm that daily FX observations were accounted for, the export rows,
keys, values and flags were preserved, no FX matches were lost, and the main table has
one row per study month. Also report the available MoM pairs and export YoY values.
The first MoM and the first 12 YoY values remain missing because no earlier history is
included; they are not replaced with zero.


In [21]:
source_columns = p3_keys + ['export_value_cad_k', 'status']
transformation_checks = pd.DataFrame([
    ('All daily FX observations counted', fx_monthly['fx_observation_count'].sum() == len(p3_fx)),
    ('Export row count preserved', len(exports_monthly) == len(p3_exports)),
    ('Export keys, amounts and flags preserved', exports_monthly[source_columns].equals(p3_exports[source_columns])),
    ('No unmatched FX months', exports_monthly['usd_cad_monthly_avg'].notna().all()),
    ('Main table has the complete monthly calendar', pd.DatetimeIndex(analysis_monthly['month']).equals(expected_months)),
], columns=['check', 'passed'])
display(transformation_checks)
assert transformation_checks['passed'].all(), 'Review failed checks before using the outputs.'

print(f"Main table: {len(analysis_monthly)} months")
print(f"Usable MoM pairs: {analysis_monthly['mom_pair_available'].sum()}")
print(f"Usable export YoY values: {analysis_monthly['exports_yoy_pct'].notna().sum()}")


,check,passed
0,All daily FX observations counted,True
1,Export row count preserved,True
2,"Export keys, amounts and flags preserved",True
3,No unmatched FX months,True
4,Main table has the complete monthly calendar,True


Main table: 96 months
Usable MoM pairs: 95
Usable export YoY values: 84


### 3.6 Save intermediate outputs for Part 4

Save under `data/transformed/`, leaving Part 2's `data/clean/` files intact. These are
local intermediate outputs; Part 4 can use the same in-memory DataFrames to build and
store the final dataset. CSV dates use `YYYY-MM-DD`; empty change cells represent
undefined values, not zero. Re-import dates with `parse_dates=['month']`.


In [22]:
TRANSFORMED_DIR = Path(ROOT) / 'data' / 'transformed'
TRANSFORMED_DIR.mkdir(parents=True, exist_ok=True)
p3_outputs = {
    'fx_monthly.csv': fx_monthly,
    'exports_monthly.csv': exports_monthly,
    'analysis_monthly.csv': analysis_monthly,
    'transformation_checks.csv': transformation_checks,
}
for filename, frame in p3_outputs.items():
    frame.to_csv(TRANSFORMED_DIR / filename, index=False, date_format='%Y-%m-%d', na_rep='')
    print(f"Saved data/transformed/{filename}: {len(frame):,} rows")


Saved data/transformed/fx_monthly.csv: 96 rows
Saved data/transformed/exports_monthly.csv: 768 rows
Saved data/transformed/analysis_monthly.csv: 96 rows
Saved data/transformed/transformation_checks.csv: 5 rows


#### Hand-off to Part 4 (Emily) and Part 1 (Rita)

| DataFrame | Current rows | Grain / use |
|---|---:|---|
| `fx_monthly` | 96 | One row per month; monthly mean FX, count of published daily observations, FX MoM |
| `exports_monthly` | 768 | One row per month–product–destination; C12 plus seven sub-groups, with FX and changes |
| `analysis_monthly` | 96 | One row per month; C12 energy total to the U.S., the main table for Q1–Q3 |
| `transformation_checks` | 5 | One row per validation check |

**New variables for the data dictionary**

| Variable | Type | Definition / unit |
|---|---|---|
| `usd_cad_monthly_avg` | Float | Arithmetic mean of published daily FXUSDCAD rates; CAD per USD |
| `fx_observation_count` | Integer | Number of published daily rates used in that month's mean |
| `fx_mom_pct` | Float, nullable | Month-over-month change in monthly mean FX, multiplied by 100 |
| `exports_mom_pct` | Float, nullable | Month-over-month change in export value within each product–destination series, multiplied by 100 |
| `exports_yoy_pct` | Float, nullable | Change from the same month one year earlier within each series, multiplied by 100 |
| `mom_pair_available` | Boolean | True when both FX MoM and export MoM are available for that row |
| `trade_type` | Text | Part 2 selection: Export |
| `country_of_origin` | Text | Part 2 selection: Canada |

Use `analysis_monthly` for the primary charts and association analysis. Use the seven
sub-groups separately when examining composition; do not sum all rows including C12.
The first month's levels remain useful for Q1/Q2 even though its MoM pair is unavailable.

**Interpretation:** These are nominal CAD export values, not export quantities.
Currency conversion, commodity prices, quantities and product composition can all affect
their changes. Monthly correlation is descriptive and does not establish causation or
isolate any one of those effects. Part 3 prepares the inputs; it does not estimate a causal
exchange-rate effect or recommend a firm-level hedge.


---
## Part 4 – Prepared dataset and data storage 

### Part 4 Overview

The prepared table already exists from Part 3. The cells below reopen it and check its structure without rewriting any data.

**Need the data without this project?** [Download the CSV from Drive](https://drive.google.com/drive/folders/1cITtAQFp08k3Eg16VB2h84qYlul-cGkP?usp=sharing) and open it in Excel, or follow the MotherDuck SQL steps in the [data access guide](../docs/datastorage/Data_Access.md). Choose one route; neither requires running this notebook.

Supporting files: [report Sections 5–6](../docs/datastorage/Sections_5_and_6.md), [access guide](../docs/datastorage/Data_Access.md), [optional local SQL](../docs/datastorage/query_prepared_data.sql), and [verification record](../docs/datastorage/Workflow_and_Verification.md).

### 4.1 Reopen the saved prepared table


In [1]:
# 4.1 Load the prepared dataset saved by Part 3.6 (read only; nothing is written)
from pathlib import Path
import pandas as pd

P4_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
P4_CSV = P4_ROOT / "data" / "transformed" / "analysis_monthly.csv"
assert P4_CSV.exists(), f"Prepared CSV not found: {P4_CSV}"

prepared = pd.read_csv(P4_CSV, parse_dates=["month"], dtype={"status": "string"})
print(f"Read {P4_CSV.relative_to(P4_ROOT).as_posix()}: {prepared.shape[0]} rows x {prepared.shape[1]} columns")
print(f"Months: {prepared['month'].min():%B %Y} to {prepared['month'].max():%B %Y}")
prepared.dtypes.astype(str).rename("dtype").to_frame()

Read data/transformed/analysis_monthly.csv: 96 rows x 14 columns
Months: January 2018 to December 2025


,dtype
month,datetime64[us]
product_code,str
product_name,str
destination,str
export_value_cad_k,int64
status,string
usd_cad_monthly_avg,float64
fx_observation_count,int64
fx_mom_pct,float64
trade_type,str


### 4.2 Structure checks

The expected values are the current snapshot of Part 3's output (96 months, 14 columns, 95 paired
month-over-month changes, 84 export year-over-year changes). A mismatch is **reported, not fixed**:
the data are never edited to force these counts. One row per month means `month` is the table's key.

In [2]:
# 4.2 Compare the prepared table with the expected snapshot (report only; data unchanged)
p4_months = pd.DatetimeIndex(prepared["month"])
p4_calendar = pd.date_range("2018-01-01", "2025-12-01", freq="MS")
p4_paired = prepared[["fx_mom_pct", "exports_mom_pct"]].notna().all(axis=1)
p4_selection = ((prepared["trade_type"] == "Export") & (prepared["country_of_origin"] == "Canada")
                & (prepared["destination"] == "United States") & (prepared["product_code"] == "C12"))

p4_checks = pd.DataFrame([
    ("Rows (months)", 96, len(prepared)),
    ("Columns", 14, prepared.shape[1]),
    ("Duplicate months (month is the unique key)", 0, int(p4_months.duplicated().sum())),
    ("Rows in month order", True, bool(p4_months.is_monotonic_increasing)),
    ("Calendar months missing, Jan 2018 - Dec 2025", 0, len(p4_calendar.difference(p4_months))),
    ("Months outside the study calendar", 0, len(p4_months.difference(p4_calendar))),
    ("Rows outside Export / Canada / United States / C12", 0, int((~p4_selection).sum())),
    ("Missing export values (export_value_cad_k)", 0, int(prepared["export_value_cad_k"].isna().sum())),
    ("Missing monthly FX averages (usd_cad_monthly_avg)", 0, int(prepared["usd_cad_monthly_avg"].isna().sum())),
    ("Paired MoM changes (fx_mom_pct and exports_mom_pct)", 95, int(p4_paired.sum())),
    ("Rows where mom_pair_available disagrees with the pair", 0, int((p4_paired != prepared["mom_pair_available"]).sum())),
    ("Export YoY changes available (exports_yoy_pct)", 84, int(prepared["exports_yoy_pct"].notna().sum())),
], columns=["check", "expected", "actual"])
p4_checks["matches"] = [e == a for e, a in zip(p4_checks["expected"], p4_checks["actual"])]
display(p4_checks)

p4_mismatches = p4_checks.loc[~p4_checks["matches"], "check"].tolist()
if p4_mismatches:
    print("DISCREPANCY - review with the Part 3 owner before use (the data were not changed):", p4_mismatches)
else:
    print("All 12 checks match the expected snapshot.")

,check,expected,actual,matches
0,Rows (months),96,96,True
1,Columns,14,14,True
2,Duplicate months (month is the unique key),0,0,True
3,Rows in month order,True,True,True
4,"Calendar months missing, Jan 2018 - Dec 2025",0,0,True
5,Months outside the study calendar,0,0,True
6,Rows outside Export / Canada / United States /...,0,0,True
7,Missing export values (export_value_cad_k),0,0,True
8,Missing monthly FX averages (usd_cad_monthly_avg),0,0,True
9,Paired MoM changes (fx_mom_pct and exports_mom...,95,95,True


All 12 checks match the expected snapshot.


### 4.3 First five months

Units used in the preview:

- `export_value_cad_k`: export value in **thousands of current Canadian dollars**
  (7,772,901 = CAD 7.77 billion). Values are nominal, not quantities.
- `usd_cad_monthly_avg`: monthly mean of daily Bank of Canada rates, in **CAD per USD**.
  A higher value means a weaker Canadian dollar.
- `fx_mom_pct`, `exports_mom_pct`: month-over-month change in **percent** (`1.28` = 1.28%).

January 2018 has no change because December 2017 is outside the file; the cell is empty (`NaN`),
not zero. For the same reason the first 12 months have no year-over-year change. Values are
rounded **for display only**; `prepared` keeps full precision.

In [3]:
# 4.3 First five months in order; rounding applies to this display copy only
p4_preview_cols = ["month", "export_value_cad_k", "usd_cad_monthly_avg", "fx_mom_pct", "exports_mom_pct"]
p4_preview = prepared.sort_values("month").head(5)[p4_preview_cols].copy()
p4_preview["month"] = p4_preview["month"].dt.strftime("%Y-%m")
p4_preview = p4_preview.round({"usd_cad_monthly_avg": 4, "fx_mom_pct": 2, "exports_mom_pct": 2})
p4_preview

,month,export_value_cad_k,usd_cad_monthly_avg,fx_mom_pct,exports_mom_pct
0,2018-01,10020138,1.2427,NaN,NaN
1,2018-02,7772901,1.2586,1.28,-22.43
2,2018-03,8436578,1.2932,2.75,8.54
3,2018-04,8343371,1.2733,-1.54,-1.10
4,2018-05,9491938,1.2873,1.11,13.77


### 4.4 Storage and retrieval

**Reference dataset:** `analysis_monthly.csv`, 96 months × 14 columns, January 2018 to December 2025.
It is available through two access routes, matching report Section 6. Choose **one** method; neither
requires downloading this repository or running this notebook.

#### Option A: Download the CSV from Google Drive (simplest)

1. Open the [Google Drive folder](https://drive.google.com/drive/folders/1cITtAQFp08k3Eg16VB2h84qYlul-cGkP?usp=sharing).
2. Select **analysis_monthly.csv** and choose **Download**.
3. Open the downloaded file in Excel. You do not need the repository first.

Optional Python access directly from Drive (requires pandas):

```python
import pandas as pd

file_id = "1XaqURO3VLrchvkVwrtXKZE7qfw-4Dgyy"
url = "https://drive.google.com/uc?export=download&id=" + file_id
df = pd.read_csv(url, parse_dates=["month"])
print(df.shape)  # (96, 14): every row and column is loaded
print(df.head(5))  # preview only; df still contains all 96 months
```

#### Option B: Query the same table in MotherDuck

Paste the SQL below into MotherDuck's web SQL editor ([app.motherduck.com](https://app.motherduck.com/)).
Sign in with your **own** MotherDuck account, using the same cloud region as the share owner.

```sql
ATTACH
 'md:_share/milestone2/707b0003-257d-4b21-8bec-e70a4b2ee441'
AS milestone2;

SELECT *
FROM milestone2.main.analysis_monthly
ORDER BY month;
```

Run ATTACH once. If `milestone2` is already attached, run only the SELECT.
The SELECT retrieves all rows and all columns; a preview is optional (add `LIMIT 5` before its final semicolon).

These cloud examples are documentation only: running Part 4 does not connect to Google Drive or MotherDuck.
Full steps and checks are in the [data access guide](../docs/datastorage/Data_Access.md) and
[report Sections 5–6](../docs/datastorage/Sections_5_and_6.md).

#### Optional local SQL for readers who already have the project

**Reference copy:** `data/transformed/analysis_monthly.csv` in this repository (96 rows, all 14
columns, full precision). It opens directly in Excel; no cloud account, token or code is needed.

The cell below shows SQL access with a local, in-memory DuckDB connection that reads the same CSV.
It needs only the `duckdb` package listed in `requirements.txt`, not a MotherDuck token. It then
retrieves all columns and all months and checks that DuckDB and pandas return identical values.
More query examples are in [`docs/datastorage/query_prepared_data.sql`](../docs/datastorage/query_prepared_data.sql)
and [`docs/datastorage/Data_Access.md`](../docs/datastorage/Data_Access.md).

In [4]:
# 4.4 Local SQL retrieval with DuckDB (in-memory; reads the CSV, writes nothing, no token needed)
import duckdb

p4_con = duckdb.connect()
p4_src = "read_csv('" + P4_CSV.as_posix().replace("'", "''") + "', header = true)"

display(p4_con.sql(f'''
    SELECT COUNT(*)                                     AS row_count,
           COUNT(DISTINCT month)                        AS distinct_months,
           MIN(month)                                   AS first_month,
           MAX(month)                                   AS last_month,
           COUNT(*) FILTER (WHERE mom_pair_available)   AS paired_mom_changes,
           COUNT(exports_yoy_pct)                       AS export_yoy_changes,
           MAX(export_value_cad_k) FILTER (WHERE month = DATE '2018-02-01') AS feb_2018_export_cad_k
    FROM {p4_src}
''').df())

# All columns and all months, then an exact comparison with the pandas table from 4.1
p4_sql_full = p4_con.sql(f"SELECT * FROM {p4_src} ORDER BY month").df()
p4_con.close()
p4_sql_full["month"] = pd.to_datetime(p4_sql_full["month"])
p4_sql_full["status"] = p4_sql_full["status"].astype("string")
try:
    pd.testing.assert_frame_equal(p4_sql_full, prepared, check_dtype=False, check_exact=True)
    print(f"DuckDB and pandas return the same table: {p4_sql_full.shape[0]} rows x {p4_sql_full.shape[1]} columns, identical values.")
except AssertionError as err:
    print("DISCREPANCY between DuckDB and pandas retrieval:", err)

,row_count,distinct_months,first_month,last_month,paired_mom_changes,export_yoy_changes,feb_2018_export_cad_k
0,96,96,2018-01-01,2025-12-01,95,84,7772901


DuckDB and pandas return the same table: 96 rows x 14 columns, identical values.


### Hand-off for Milestone 3

Use the existing `analysis_monthly.csv` table for the three research questions. There are 96 monthly levels, 95 paired monthly changes and 84 export YoY changes.

For any immediate access, [download the CSV from Drive](https://drive.google.com/drive/folders/1cITtAQFp08k3Eg16VB2h84qYlul-cGkP?usp=sharing) or use the **milestone2** MotherDuck share described in the [data access guide](../docs/datastorage/Data_Access.md). Local SQL is an optional alternative for readers who have downloaded the project.

Only Part 4 needs to run to inspect this saved data. Parts 2 and 3 remain unchanged.
